In [0]:
TEAM_NAME   = "team-4"
CATALOG     = "de_workspace26"
YOUR_DB     = f"{CATALOG}.quickmeds_team_4"
 
# S3 paths
BASE_PATH   = "s3://s3-de-q1-26/DE-Training/quickmeds-datalake-team-4/"
RAW_PATH    = f"{BASE_PATH}raw/"
BRONZE_PATH = f"{BASE_PATH}bronze/"
 
# Raw source locations (CSVs uploaded here from source system)
RAW_CUSTOMERS   = f"{RAW_PATH}customers/"
RAW_ORDERS      = f"{RAW_PATH}orders/"
RAW_ORDER_ITEMS = f"{RAW_PATH}order_items/"
RAW_DELIVERIES  = f"{RAW_PATH}deliveries/"
RAW_PRODUCTS    = f"{RAW_PATH}products/"
 
# Bronze Delta table output locations 
BRONZE_CUSTOMERS   = f"{BRONZE_PATH}customers_dbx/"
BRONZE_ORDERS      = f"{BRONZE_PATH}orders_dbx/"
BRONZE_ORDER_ITEMS = f"{BRONZE_PATH}order_items_dbx/"
BRONZE_DELIVERIES  = f"{BRONZE_PATH}deliveries_dbx/"
BRONZE_PRODUCTS    = f"{BRONZE_PATH}products_dbx/"
 
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {YOUR_DB}")
 
print("=" * 58)
print(f"  Project  : QuickMeds Data Platform")
print(f"  Team     : {TEAM_NAME}")
print(f"  Catalog  : {CATALOG}")
print(f"  Schema   : {YOUR_DB}")
print(f"  Raw S3   : {RAW_PATH}")
print(f"  Bronze S3: {BRONZE_PATH}")
print("=" * 58)
print("Ready — run cells in order from Cell 1")

In [0]:
from pyspark.sql import functions as F
from datetime import date
 
RUN_ID = str(date.today())   # e.g. "2026-03-27"
 
print(f"Imports loaded | RUN_ID = {RUN_ID}")

In [0]:
df_customers_raw = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(RAW_CUSTOMERS)
)
 
df_customers_bronze = (
    df_customers_raw
        .withColumn("_source",     F.lit("s3_raw_customers"))
        .withColumn("_ingest_ts",  F.current_timestamp())
        .withColumn("_file_name",  F.lit(RAW_CUSTOMERS))
        .withColumn("_run_id",     F.lit(RUN_ID))
        .withColumn("ingest_date", F.current_date())
)
 
spark.sql(f"DROP TABLE IF EXISTS {YOUR_DB}.customers_bronze")
 
(
    df_customers_bronze.write
        .format("delta")
        .mode("append")
        .partitionBy("ingest_date")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("path", BRONZE_CUSTOMERS)
        .saveAsTable(f"{YOUR_DB}.customers_bronze")
)
 
count = spark.table(f"{YOUR_DB}.customers_bronze").count()
print(f"customers_bronze written: {count} rows")
print(f"Location: {BRONZE_CUSTOMERS}")

In [0]:
spark.sql(f"""
    ALTER TABLE {YOUR_DB}.customers_bronze
    ADD CONSTRAINT customers_id_not_null
    CHECK (customer_id IS NOT NULL)
""")
 
print("Constraints added to customers_bronze")
print("  customers_id_not_null")

In [0]:
df_orders_raw = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(RAW_ORDERS)
)
 
df_orders_bronze = (
    df_orders_raw
        .withColumn("_source",     F.lit("s3_raw_orders"))
        .withColumn("_ingest_ts",  F.current_timestamp())
        .withColumn("_file_name",  F.lit(RAW_ORDERS))
        .withColumn("_run_id",     F.lit(RUN_ID))
        .withColumn("ingest_date", F.current_date())
)
 
spark.sql(f"DROP TABLE IF EXISTS {YOUR_DB}.orders_bronze")
 
(
    df_orders_bronze.write
        .format("delta")
        .mode("append")
        .partitionBy("ingest_date")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("path", BRONZE_ORDERS)
        .saveAsTable(f"{YOUR_DB}.orders_bronze")
)
 
count = spark.table(f"{YOUR_DB}.orders_bronze").count()
print(f"orders_bronze written: {count} rows")
print(f"Location: {BRONZE_ORDERS}")

In [0]:
spark.sql(f"""
    ALTER TABLE {YOUR_DB}.orders_bronze
    ADD CONSTRAINT orders_id_not_null
    CHECK (order_id IS NOT NULL)
""")
 
spark.sql(f"""
    ALTER TABLE {YOUR_DB}.orders_bronze
    ADD CONSTRAINT orders_total_non_negative
    CHECK (total_amount >= 0 OR total_amount IS NULL)
""")
 
print("Constraints added to orders_bronze")
print("  orders_id_not_null")
print("  orders_total_non_negative")

In [0]:
df_order_items_raw = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(RAW_ORDER_ITEMS)
)
 
df_order_items_bronze = (
    df_order_items_raw
        .withColumn("_source",     F.lit("s3_raw_order_items"))
        .withColumn("_ingest_ts",  F.current_timestamp())
        .withColumn("_file_name",  F.lit(RAW_ORDER_ITEMS))
        .withColumn("_run_id",     F.lit(RUN_ID))
        .withColumn("ingest_date", F.current_date())
)
 
spark.sql(f"DROP TABLE IF EXISTS {YOUR_DB}.order_items_bronze")
 
(
    df_order_items_bronze.write
        .format("delta")
        .mode("append")
        .partitionBy("ingest_date")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("path", BRONZE_ORDER_ITEMS)
        .saveAsTable(f"{YOUR_DB}.order_items_bronze")
)
 
count = spark.table(f"{YOUR_DB}.order_items_bronze").count()
print(f"order_items_bronze written: {count} rows")
print(f"Location: {BRONZE_ORDER_ITEMS}")

In [0]:
spark.sql(f"""
    ALTER TABLE {YOUR_DB}.order_items_bronze
    ADD CONSTRAINT items_id_not_null
    CHECK (item_id IS NOT NULL)
""")
 
spark.sql(f"""
    ALTER TABLE {YOUR_DB}.order_items_bronze
    ADD CONSTRAINT items_quantity_positive
    CHECK (quantity > 0 OR quantity IS NULL)
""")
 
spark.sql(f"""
    ALTER TABLE {YOUR_DB}.order_items_bronze
    ADD CONSTRAINT items_unit_price_non_negative
    CHECK (unit_price >= 0 OR unit_price IS NULL)
""")
 
print("Constraints added to order_items_bronze")
print("  items_id_not_null")
print("  items_quantity_positive")
print("  items_unit_price_non_negative")

In [0]:
df_deliveries_raw = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(RAW_DELIVERIES)
)
 
df_deliveries_bronze = (
    df_deliveries_raw
        .withColumn("_source",     F.lit("s3_raw_deliveries"))
        .withColumn("_ingest_ts",  F.current_timestamp())
        .withColumn("_file_name",  F.lit(RAW_DELIVERIES))
        .withColumn("_run_id",     F.lit(RUN_ID))
        .withColumn("ingest_date", F.current_date())
)
 
spark.sql(f"DROP TABLE IF EXISTS {YOUR_DB}.deliveries_bronze")
 
(
    df_deliveries_bronze.write
        .format("delta")
        .mode("append")
        .partitionBy("ingest_date")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("path", BRONZE_DELIVERIES)
        .saveAsTable(f"{YOUR_DB}.deliveries_bronze")
)
 
count = spark.table(f"{YOUR_DB}.deliveries_bronze").count()
print(f"deliveries_bronze written: {count} rows")
print(f"Location: {BRONZE_DELIVERIES}")

In [0]:
spark.sql(f"""
    ALTER TABLE {YOUR_DB}.deliveries_bronze
    ADD CONSTRAINT deliveries_id_not_null
    CHECK (delivery_id IS NOT NULL)
""")
 
spark.sql(f"""
    ALTER TABLE {YOUR_DB}.deliveries_bronze
    ADD CONSTRAINT deliveries_minutes_positive
    CHECK (delivery_minutes > 0 OR delivery_minutes IS NULL)
""")
 
spark.sql(f"""
    ALTER TABLE {YOUR_DB}.deliveries_bronze
    ADD CONSTRAINT deliveries_rating_range
    CHECK (rating BETWEEN 1.0 AND 5.0 OR rating IS NULL)
""")
 
print("Constraints added to deliveries_bronze")
print("  deliveries_id_not_null")
print("  deliveries_minutes_positive")
print("  deliveries_rating_range")

In [0]:
df_products_raw = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(RAW_PRODUCTS)
)
 
df_products_bronze = (
    df_products_raw
        .withColumn("_source",     F.lit("s3_raw_products"))
        .withColumn("_ingest_ts",  F.current_timestamp())
        .withColumn("_file_name",  F.lit(RAW_PRODUCTS))
        .withColumn("_run_id",     F.lit(RUN_ID))
        .withColumn("ingest_date", F.current_date())
)
 
spark.sql(f"DROP TABLE IF EXISTS {YOUR_DB}.products_bronze")
 
(
    df_products_bronze.write
        .format("delta")
        .mode("append")
        .partitionBy("ingest_date")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("path", BRONZE_PRODUCTS)
        .saveAsTable(f"{YOUR_DB}.products_bronze")
)
 
count = spark.table(f"{YOUR_DB}.products_bronze").count()
print(f"products_bronze written: {count} rows")
print(f"Location: {BRONZE_PRODUCTS}")

In [0]:
spark.sql(f"""
    ALTER TABLE {YOUR_DB}.products_bronze
    ADD CONSTRAINT products_id_not_null
    CHECK (product_id IS NOT NULL)
""")
 
spark.sql(f"""
    ALTER TABLE {YOUR_DB}.products_bronze
    ADD CONSTRAINT products_price_non_negative
    CHECK (unit_price >= 0 OR unit_price IS NULL)
""")
 
print("Constraints added to products_bronze")
print("  products_id_not_null")
print("  products_price_non_negative")

In [0]:
df_bad = spark.createDataFrame(
    [("RX9999", "UNEXPECTED_VALUE")],
    ["order_id", "UNEXPECTED_COLUMN"]
)
 
try:
    df_bad.write.format("delta").mode("append") \
        .saveAsTable(f"{YOUR_DB}.orders_bronze")
    print("ERROR: Should have been rejected!")
except Exception as e:
    print("Schema enforcement WORKS — bad write correctly rejected")
    print(f"Reason: {str(e)[:150]}")

In [0]:
def bronze_validator(df, table_name, key_column, expected_columns):
    """
    Validates a Bronze DataFrame against 4 quality rules.
    Returns results dict. Assert on results in Workflow for alerting.
    """
    results   = {}
    row_count = df.count()
 
    # Rule 1 — table must have rows
    results["row_count"] = "pass" if row_count > 0 else "fail"
    print(f"  row_count      : {row_count:>6}  -> {results['row_count']}")
 
    # Rule 2 — key column null % must be under 5 %
    null_count = df.filter(F.col(key_column).isNull()).count()
    null_pct   = (null_count / row_count * 100) if row_count > 0 else 100.0
    results["null_check"] = {
        "status"    : "pass" if null_pct < 5 else "fail",
        "null_count": null_count,
        "null_pct"  : round(null_pct, 2)
    }
    print(f"  null_check     : {null_count} nulls ({null_pct:.1f}%)  -> {results['null_check']['status']}")
 
    # Rule 3 — all expected source columns must be present
    missing_cols = [c for c in expected_columns if c not in df.columns]
    results["schema_check"] = {
        "status" : "pass" if not missing_cols else "fail",
        "missing": missing_cols
    }
    print(f"  schema_check   : missing={missing_cols}  -> {results['schema_check']['status']}")
 
    # Rule 4 — all 4 Bronze metadata columns must exist
    meta_cols    = ["_source", "_ingest_ts", "_file_name", "_run_id"]
    missing_meta = [c for c in meta_cols if c not in df.columns]
    results["metadata_check"] = "pass" if not missing_meta else "fail"
    print(f"  metadata_check : missing={missing_meta}  -> {results['metadata_check']}")
 
    overall = "PASS" if all([
        results["row_count"]              == "pass",
        results["null_check"]["status"]   == "pass",
        results["schema_check"]["status"] == "pass",
        results["metadata_check"]         == "pass"
    ]) else "FAIL"
 
    print(f"\n  [{table_name}]  Overall -> {overall}\n")
    return results
 
 
print("=" * 58)
print("  BRONZE VALIDATION REPORT — QuickMeds Team 4")
print("=" * 58)
 
print("\n-- customers_bronze --")
r_cust = bronze_validator(
    spark.table(f"{YOUR_DB}.customers_bronze"),
    "customers_bronze", "customer_id",
    ["customer_id", "name", "city", "signup_date",
     "membership_tier", "email", "phone"]
)
assert r_cust["row_count"] == "pass", "customers_bronze row count check FAILED"
 
print("-- orders_bronze --")
r_ord = bronze_validator(
    spark.table(f"{YOUR_DB}.orders_bronze"),
    "orders_bronze", "order_id",
    ["order_id", "customer_id", "order_date", "order_time",
     "status", "total_amount", "payment_method", "city"]
)
assert r_ord["row_count"] == "pass", "orders_bronze row count check FAILED"
 
print("-- order_items_bronze --")
r_itm = bronze_validator(
    spark.table(f"{YOUR_DB}.order_items_bronze"),
    "order_items_bronze", "item_id",
    ["item_id", "order_id", "product_id", "quantity",
     "unit_price", "line_total"]
)
assert r_itm["row_count"] == "pass", "order_items_bronze row count check FAILED"
 
print("-- deliveries_bronze --")
r_del = bronze_validator(
    spark.table(f"{YOUR_DB}.deliveries_bronze"),
    "deliveries_bronze", "delivery_id",
    ["delivery_id", "order_id", "agent_id", "agent_name",
     "city", "pickup_time", "drop_time",
     "delivery_minutes", "distance_km", "rating"]
)
assert r_del["row_count"] == "pass", "deliveries_bronze row count check FAILED"
 
print("-- products_bronze --")
r_prd = bronze_validator(
    spark.table(f"{YOUR_DB}.products_bronze"),
    "products_bronze", "product_id",
    ["product_id", "product_name", "category", "unit_price",
     "unit", "manufacturer", "prescription_required", "in_stock"]
)
assert r_prd["row_count"] == "pass", "products_bronze row count check FAILED"
 
print("=" * 58)
print("  ALL 5 BRONZE TABLES VALIDATED SUCCESSFULLY")
print("=" * 58)

In [0]:
tables = {
    "customers_bronze"  : "customer_id",
    "orders_bronze"     : "order_id",
    "order_items_bronze": "item_id",
    "deliveries_bronze" : "delivery_id",
    "products_bronze"   : "product_id",
}
 
print("=" * 58)
print("  BRONZE LAYER — FINAL SUMMARY")
print(f"  RUN_ID     : {RUN_ID}")
print(f"  Bronze S3  : {BRONZE_PATH}")
print("=" * 58)
 
for tbl, key in tables.items():
    df         = spark.table(f"{YOUR_DB}.{tbl}")
    cnt        = df.count()
    partitions = df.select("ingest_date").distinct().count()
    print(f"  {tbl:<25} | rows: {cnt:>5} | partitions: {partitions}")
 
print("=" * 58)
print("  Bronze layer complete — ready for Silver ingestion")
print("=" * 58)
 
# Spot-check: sample rows from orders_bronze
print("\nSample rows from orders_bronze:")
spark.table(f"{YOUR_DB}.orders_bronze").select(
    "order_id", "customer_id", "city", "status",
    "total_amount", "_source", "_ingest_ts", "_run_id"
).show(3, truncate=False)